# Practice Exercise 4


## Step 1: Identify the Task

We want our self-driving trailer to find a route from Nairobi to Kisumu and know how to react to different road conditions along the way.

**Goal:** Get from Nairobi to Kisumu using the shortest path (370 km: Nairobi -> Naivasha -> Nakuru -> Molo -> Kericho -> Kisumu) while staying safe and handling road issues.

## Step 2: Assemble Relevant Knowledge

From Exercise 3, we have:
- The highway map with towns and distances in km (Nairobi, Naivasha, Nakuru, Molo, Kericho, Kisumu, Narok, Bomet, Eldoret, Kapsabet, etc.).
- Environment notes: the real road is dynamic, partially observable, and multi-agent, while the map search is static and discrete.
- Basic driving actions for road conditions like fog, rain, potholes, traffic jams, and roadblocks.

We didn't use any external APIs or web scraping here, everything is stored locally in dictionaries in the notebook.

## Step 3: Define the Vocabulary

- **Towns (nodes):** Nairobi, Ruiru, Thika, Embu, Meru, Nanyuki, Nyeri, Nyahururu, Eldama Ravine, Nakuru, Naivasha, Narok, Bomet, Kisii, Kericho, Molo, Eldoret, Kapsabet, Webuye, Kakamega, Kisumu.
- **Road conditions:** Clear, Fog, Potholes, Traffic Jam, Roadblock, Heavy Rain (plus any new ones we add).
- **Main functions we use:** `ucs_shortest_path(graph, start, goal)` for finding the shortest route, and `query_trailer_agent(...)` to check the route and driving advice.

## Step 4: Encode General Knowledge about the Domain


In [ ]:
import heapq

# Graph from exercise 3: town -> list of (next_town, distance_km)
highway_kb = {
    "Nairobi": [("Ruiru", 20), ("Naivasha", 90)],
    "Ruiru": [("Thika", 25)],
    "Thika": [("Nyeri", 110), ("Embu", 85)],
    "Embu": [("Meru", 75), ("Thika", 85)],
    "Meru": [("Nanyuki", 75)],
    "Nanyuki": [("Nyeri", 55)],
    "Nyeri": [("Nairobi", 115), ("Nyahururu", 80), ("Nanyuki", 55)],
    "Nyahururu": [("Nakuru", 105), ("Eldama Ravine", 70), ("Nyeri", 80)],
    "Eldama Ravine": [("Eldoret", 90), ("Nyahururu", 70)],
    "Nakuru": [
        ("Naivasha", 65),
        ("Nyahururu", 105),
        ("Eldama Ravine", 70),
        ("Eldoret", 155),
        ("Molo", 50),
        ("Nyeri", 185),
    ],
    "Naivasha": [("Nakuru", 65), ("Narok", 100), ("Nairobi", 90)],
    "Narok": [("Naivasha", 100), ("Bomet", 85)],
    "Bomet": [("Narok", 85), ("Kericho", 60), ("Kisii", 75)],
    "Kisii": [("Bomet", 75), ("Kisumu", 115)],
    "Kericho": [("Bomet", 60), ("Molo", 65), ("Kisumu", 100)],
    "Molo": [("Nakuru", 50), ("Kericho", 65)],
    "Eldoret": [("Webuye", 75), ("Kapsabet", 40), ("Eldama Ravine", 90), ("Nakuru", 155)],
    "Kapsabet": [("Eldoret", 40), ("Kisumu", 120)],
    "Webuye": [("Eldoret", 75), ("Kakamega", 40)],
    "Kakamega": [("Webuye", 40), ("Kisumu", 55)],
    "Kisumu": [("Kakamega", 55), ("Kapsabet", 120), ("Kericho", 100), ("Kisii", 115)],
}

# Actions for different road conditions
road_rules_kb = {
    "Clear": "Keep steady speed and stay in lane.",
    "Fog": "Turn on fog lights, slow down, and keep extra distance.",
    "Potholes": "Slow down so the trailer and cargo don't take damage.",
    "Traffic Jam": "Wait or check another route if the delay is too long.",
    "Roadblock": "Stop safely and find another route.",
    "Heavy Rain": "Slow down, turn on lights and wipers, and brake gently.",
}

## Step 5: Add New Facts to the Knowledge Base


In [ ]:
def show_kb():
    print("\n--- Road Rules ---")
    for cond, action in road_rules_kb.items():
        print(f"{cond:16} -> {action}")

    print("\n--- Sample Roads ---")
    for town in ["Nairobi", "Naivasha", "Nakuru", "Molo", "Kericho"]:
        print(f"{town:10} -> {highway_kb.get(town, [])}")


def add_road_fact(origin, destination, distance_km):
    highway_kb.setdefault(origin, [])
    highway_kb[origin] = [(nxt, d) for nxt, d in highway_kb[origin] if nxt != destination]
    highway_kb[origin].append((destination, distance_km))
    print(f"Added road: {origin} -> {destination} ({distance_km} km)")


def add_condition_fact(condition, action):
    road_rules_kb[condition.title()] = action
    print(f"Added condition: '{condition.title()}' -> '{action}'")


add_condition_fact("Escarpment Wind", "Slow down and keep a firm hold on steering on windy stretches.")
show_kb()

## Steps 6 & 7: Encode Specific Problem Instance & Pose Queries


In [ ]:
def ucs_shortest_path(graph, start, goal):
    frontier = [(0, start, [start])]
    explored = set()
    best_cost = {start: 0}

    while frontier:
        cost, node, path = heapq.heappop(frontier)
        if node in explored:
            continue
        if node == goal:
            return path, cost

        explored.add(node)
        for nxt, weight in graph.get(node, []):
            new_cost = cost + weight
            if nxt not in explored and (nxt not in best_cost or new_cost < best_cost[nxt]):
                best_cost[nxt] = new_cost
                heapq.heappush(frontier, (new_cost, nxt, path + [nxt]))

    return None, float("inf")


def query_trailer_agent(current_town="Nairobi", goal_town="Kisumu", road_condition="Clear"):
    start = current_town.strip().title()
    goal = goal_town.strip().title()
    cond = road_condition.strip().title()

    at_goal = (start == goal)
    advice = road_rules_kb.get(cond, "Unknown condition — slow down and be careful.")
    path, cost = ucs_shortest_path(highway_kb, start, goal)

    print(f"Location : {start}")
    print(f"Goal     : {goal}")
    print(f"At Goal  : {at_goal}")
    print(f"Condition: {cond}")
    print(f"Advice   : {advice}")
    if path:
        print(f"Route    : {' -> '.join(path)} ({cost} km)")
    else:
        print("Route    : No path found")


query_trailer_agent(current_town="Nairobi", goal_town="Kisumu", road_condition="Fog")

## Step 8: Querying the Knowledge Base


In [ ]:
for town, cond in [
    ("Nairobi", "Clear"),
    ("Nakuru", "Potholes"),
    ("Kericho", "Heavy Rain"),
    ("Kisumu", "Clear"),
]:
    query_trailer_agent(town, "Kisumu", cond)
    print()

## Step 9: Debugging the Knowledge Base


In [ ]:
highway_kb["DeadEnd"] = []

debug_queries = [
    ("Nairobi", "Kisumu", "Clear"),
    ("Nairobi", "Kisumu", "Slippery Surface"),
    ("DeadEnd", "Kisumu", "Clear"),
    ("Kisumu", "Kisumu", "Clear"),
]

for start_t, goal_t, cond in debug_queries:
    path, cost = ucs_shortest_path(highway_kb, start_t, goal_t)
    action = road_rules_kb.get(cond, "Unknown condition — slow down and be careful.")
    print(f"From={start_t:10} To={goal_t:7} Cond={cond:16} -> Path={path}, Cost={cost}, Action={action}")

## Step 10: Final Reflection

- Using fixed distances only works for a static map; real traffic, weather and road delays can change the best choice on the road.
- Uniform cost search checks paths by distance alone, so it can expand extra nodes before reaching Kisumu.
- Unlisted conditions fall back to a generic caution message unless we add them to `road_rules_kb`.
- Next things we could do if we expand this: use A* search instead of UCS and factor in the actual real-time traffic or road closures instead of just distance.